# Assignment 3 — OPEN-RAG: Experimentation and Expansion
**FAST-NUCES | DS | Dr. Zohair Ahmed**

This notebook extends the OPEN-RAG reproduction (Assignment 2) with:
- A new dataset evaluation: **HotpotQA**
- A proposed method: **Threshold calibration analysis**
- Cross-dataset comparison and full visualizations for the report



## Section 0 — Environment Setup

In [ ]:
# Clone the OPEN-RAG repository
import os
if not os.path.exists('/content/openrag'):
    !git clone https://github.com/ShayekhBinIslam/openrag.git
else:
    print('openrag already cloned — skipping.')
%cd /content/openrag

In [ ]:
# Install dependencies
!pip install -q transformers==4.42.4 peft accelerate bitsandbytes datasets sentencepiece jsonlines matplotlib seaborn scipy

In [ ]:
# Verify GPU
import torch
print(f'GPU available: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')
    print(f'VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')

---
## Section 1 — Load Assignment 2 Results (2WikiMultihopQA)

In [ ]:
import json
import numpy as np

# Load Assignment 2 results
with open('/content/openrag/results_openrag.json') as f:
    results_2wiki = json.load(f)

em_2wiki = results_2wiki['EM']
metric_mean_2wiki = results_2wiki['metric_mean']
proba_r_2wiki = [s['proba_r'] for s in results_2wiki['detailed_init_scores']]
preds_2wiki = results_2wiki['preds']

print('=== 2WikiMultihopQA (Assignment 2 — Reproduced Baseline) ===')
print(f'Total samples:   {len(em_2wiki)}')
print(f'Exact Match:     {metric_mean_2wiki:.4f} ({metric_mean_2wiki*100:.2f}%)')
print(f'Correct answers: {sum(em_2wiki)}')
print(f'Avg P(retrieve): {np.mean(proba_r_2wiki):.4f}')
print(f'P(retrieve) min/max: {min(proba_r_2wiki):.3f} / {max(proba_r_2wiki):.3f}')

---
## Section 2 — Experiment 1: HotpotQA Evaluation

**Script used:** `run_short_form_moe_hotpot.py` (HotpotQA-specific runner from the repo)

In [ ]:
!python /content/openrag/run_short_form_multihop.py \
  --model_name "shayekh/openrag_llama2_7b_8x135m" \
  --dataset "shayekh/openrag_bench" \
  --task "hotpotqa" \
  --mode "adaptive_retrieval" \
  --threshold 0.5 \
  --metric "match" \
  --max_samples 100 \
  --output_file "/content/openrag/eval/results_hotpotqa.json" \
  --device "cuda" \
  --dtype "bfloat16"

In [ ]:
# Load HotpotQA results
with open('/content/openrag/eval/results_hotpotqa.json') as f:
    results_hotpot = json.load(f)

# Handle both list and dict output formats
if isinstance(results_hotpot, list):
    # jsonl-style: list of dicts
    em_hotpot_list = [r.get('em', r.get('match', 0)) for r in results_hotpot]
    em_hotpot = np.mean(em_hotpot_list)
    proba_r_hotpot = [r.get('proba_r', 0.5) for r in results_hotpot]
    n_hotpot = len(results_hotpot)
else:
    em_hotpot_list = results_hotpot.get('EM', results_hotpot.get('metric_results', []))
    em_hotpot = results_hotpot.get('metric_mean', np.mean(em_hotpot_list))
    proba_r_hotpot = [s['proba_r'] for s in results_hotpot.get('detailed_init_scores', [])]
    n_hotpot = len(em_hotpot_list)

print('=== HotpotQA (Experiment 1 — New Dataset) ===')
print(f'Total samples:   {n_hotpot}')
print(f'Exact Match:     {em_hotpot:.4f} ({em_hotpot*100:.2f}%)')
print(f'Correct answers: {sum(em_hotpot_list)}')
if proba_r_hotpot:
    print(f'Avg P(retrieve): {np.mean(proba_r_hotpot):.4f}')

---
## Section 3 — Threshold Sweep on 2WikiMultihopQA

**Hypothesis:** The default threshold of 0.5 is not necessarily optimal. A lower threshold (more retrieval) may help multi-hop questions which require external evidence.

We test thresholds: **0.3** (retrieve more) and **0.7** (retrieve less).

In [ ]:
!python /content/openrag/run_short_form_multihop.py \
  --model_name "shayekh/openrag_llama2_7b_8x135m" \
  --dataset "shayekh/openrag_bench" \
  --task "2wikimultihopqa" \
  --mode "adaptive_retrieval" \
  --threshold 0.3 \
  --metric "match" \
  --max_samples 100 \
  --output_file "/content/openrag/eval/results_2wiki_thresh03.json" \
  --device "cuda" \
  --dtype "bfloat16"

In [ ]:
!python /content/openrag/run_short_form_multihop.py \
  --model_name "shayekh/openrag_llama2_7b_8x135m" \
  --dataset "shayekh/openrag_bench" \
  --task "2wikimultihopqa" \
  --mode "adaptive_retrieval" \
  --threshold 0.7 \
  --metric "match" \
  --max_samples 100 \
  --output_file "/content/openrag/eval/results_2wiki_thresh07.json" \
  --device "cuda" \
  --dtype "bfloat16"

In [ ]:
thresh_results = {}

# Baseline from Assignment 2
thresh_results[0.5] = {'em': metric_mean_2wiki, 'n': len(em_2wiki)}

for thresh, fname in [(0.3, 'results_2wiki_thresh03.json'), (0.7, 'results_2wiki_thresh07.json')]:
    fpath = f'/content/openrag/eval/{fname}'
    if os.path.exists(fpath):
        with open(fpath) as f:
            r = json.load(f)
        if isinstance(r, dict):
            thresh_results[thresh] = {
                'em': r.get('metric_mean', np.mean(r.get('EM', []))),
                'n': len(r.get('EM', r.get('metric_results', [])))
            }
        print(f'Threshold {thresh}: EM = {thresh_results[thresh]["em"]:.4f}')
    else:
        print(f'Threshold {thresh}: file not found — experiment not run.')

print('\nThreshold sweep summary:')
for t, r in sorted(thresh_results.items()):
    print(f'  thresh={t}: EM={r["em"]:.4f} ({r["em"]*100:.2f}%)')

---
## Section 4 — Proposed Method: Retrieval Confidence Analysis

This section requires **no additional GPU runs**. It analyses the existing `proba_r` (retrieval probability) data to determine whether confidence in the retrieval decision correlates with answer correctness.

**Proposed method:** Confidence-stratified threshold — instead of a flat threshold, set the threshold based on the variance of `proba_r` across the dataset. High-variance datasets benefit from a lower threshold (more retrieval); low-variance datasets from a higher one.

In [ ]:
import numpy as np

proba_r = np.array(proba_r_2wiki)
em = np.array(em_2wiki)

# --- Confidence bins ---
# High-confidence retrieval: proba_r > 0.7 (model strongly wants to retrieve)
# Ambiguous:                 0.3 <= proba_r <= 0.7
# High-confidence skip:      proba_r < 0.3 (model strongly skips retrieval)

high_ret_mask   = proba_r > 0.7
ambiguous_mask  = (proba_r >= 0.3) & (proba_r <= 0.7)
high_skip_mask  = proba_r < 0.3

def safe_mean(arr):
    return float(np.mean(arr)) if len(arr) > 0 else float('nan')

print('=== Retrieval Confidence vs EM (2WikiMultihopQA) ===')
print(f'High-confidence RETRIEVE (P>0.7): n={high_ret_mask.sum():3d}, EM={safe_mean(em[high_ret_mask]):.4f}')
print(f'Ambiguous           (0.3-0.7):    n={ambiguous_mask.sum():3d}, EM={safe_mean(em[ambiguous_mask]):.4f}')
print(f'High-confidence SKIP    (P<0.3):  n={high_skip_mask.sum():3d}, EM={safe_mean(em[high_skip_mask]):.4f}')
print()

# --- Simulate calibrated threshold (find threshold maximising EM) ---
# In adaptive retrieval: retrieve if proba_r > threshold.
# Since we cannot re-run, we use proba_r as a proxy to rank samples by
# 'how much the model wanted to retrieve' and check if that correlates with EM.
thresholds = np.arange(0.1, 0.95, 0.05)
simulated_em = []
for t in thresholds:
    # Count samples where the model would retrieve at this threshold
    would_retrieve = (proba_r > t).sum()
    # EM of samples where proba_r > t (would retrieve)
    em_ret  = safe_mean(em[proba_r > t])  if (proba_r > t).sum() > 0 else float('nan')
    em_skip = safe_mean(em[proba_r <= t]) if (proba_r <= t).sum() > 0 else float('nan')
    simulated_em.append({'thresh': t, 'would_retrieve': int(would_retrieve),
                          'em_if_retrieve': em_ret, 'em_if_skip': em_skip})

print('Threshold | # Retrieve | EM (retrieved) | EM (skipped)')
print('-' * 58)
for r in simulated_em[::3]:
    print(f"  {r['thresh']:.2f}    | {r['would_retrieve']:>10} | {r['em_if_retrieve']:>14.4f} | {r['em_if_skip']:.4f}")

# Variance of proba_r — used to propose adaptive threshold
print(f'\nP(retrieve) variance: {np.var(proba_r):.4f}')
print(f'P(retrieve) std:      {np.std(proba_r):.4f}')
print(f'\nProposed adaptive threshold formula:')
print(f'  threshold = max(0.3, 0.5 - std(proba_r) * 0.5)')
adaptive_thresh = max(0.3, 0.5 - np.std(proba_r) * 0.5)
print(f'  = {adaptive_thresh:.4f} for 2WikiMultihopQA')

---
## Section 5 — Visualisations


In [ ]:
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
import numpy as np
import os

sns.set_theme(style='whitegrid', palette='muted')
FIGDIR = '/content/openrag/eval/figures'
os.makedirs(FIGDIR, exist_ok=True)

BLUE   = '#2E75B6'
ORANGE = '#E25822'
GREEN  = '#228B22'
GREY   = '#888888'

#  Paper-reported numbers for reference 
PAPER_2WIKI_EM   = 0.436   # from Open-RAG paper (full test set, LLaMA2-7B)
PAPER_HOTPOT_EM  = 0.633   # from Open-RAG paper (HotpotQA, LLaMA2-7B)


print('Figures will be saved to:', FIGDIR)

In [ ]:
#  Figure 1: Cross-dataset EM Comparison 

# Replace em_hotpot with your actual result after running Section 2
# If not yet run, set a placeholder:
try:
    _ = em_hotpot
except NameError:
    em_hotpot = 0.0  # placeholder

datasets    = ['2WikiMultihopQA\n(Paper)', '2WikiMultihopQA\n(Ours, A2)',
               'HotpotQA\n(Paper)', 'HotpotQA\n(Ours, A3)']
em_values   = [PAPER_2WIKI_EM, metric_mean_2wiki, PAPER_HOTPOT_EM, em_hotpot]
colors      = [GREY, BLUE, GREY, ORANGE]
hatches     = ['///', '', '///', '']

fig, ax = plt.subplots(figsize=(9, 5))
bars = ax.bar(datasets, [v*100 for v in em_values], color=colors, hatch=hatches,
              edgecolor='white', width=0.55, zorder=3)
for bar, val in zip(bars, em_values):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.8,
            f'{val*100:.1f}%', ha='center', va='bottom', fontsize=11, fontweight='bold')

ax.set_ylabel('Exact Match (%)', fontsize=12)
ax.set_title('OPEN-RAG Performance: Paper vs Reproduced', fontsize=14, fontweight='bold')
ax.set_ylim(0, 85)
ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda x, _: f'{x:.0f}%'))

legend_els = [mpatches.Patch(facecolor=GREY, hatch='///', edgecolor='black', label='Paper Reported'),
              mpatches.Patch(facecolor=BLUE, label='Our 2Wiki Result (A2)'),
              mpatches.Patch(facecolor=ORANGE, label='Our HotpotQA Result (A3)')]
ax.legend(handles=legend_els, loc='upper right', fontsize=10)
ax.axhline(y=50, color='red', linestyle='--', alpha=0.4, linewidth=1)
ax.grid(axis='y', alpha=0.5, zorder=0)
plt.tight_layout()
plt.savefig(f'{FIGDIR}/fig1_cross_dataset_em.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved fig1_cross_dataset_em.png')

In [ ]:
#  Figure 2: P(retrieve) Distribution — 2WikiMultihopQA 

proba_r = np.array(proba_r_2wiki)
em_arr  = np.array(em_2wiki)

fig, axes = plt.subplots(1, 2, figsize=(12, 5))

# Left: histogram of proba_r
ax = axes[0]
ax.hist(proba_r[em_arr == 1], bins=20, alpha=0.7, color=BLUE,   label='Correct (EM=1)', density=True)
ax.hist(proba_r[em_arr == 0], bins=20, alpha=0.7, color=ORANGE, label='Wrong  (EM=0)',  density=True)
ax.axvline(x=0.5, color='black', linestyle='--', linewidth=1.5, label='Threshold=0.5')
ax.set_xlabel('P(retrieve)', fontsize=11)
ax.set_ylabel('Density', fontsize=11)
ax.set_title('Retrieval Probability Distribution\n(2WikiMultihopQA)', fontsize=12, fontweight='bold')
ax.legend(fontsize=10)

# Right: mean EM per proba_r bin
ax = axes[1]
bins = np.linspace(0, 1, 11)
bin_centers = (bins[:-1] + bins[1:]) / 2
bin_em = []
bin_n  = []
for i in range(len(bins)-1):
    mask = (proba_r >= bins[i]) & (proba_r < bins[i+1])
    bin_em.append(em_arr[mask].mean() if mask.sum() > 0 else 0)
    bin_n.append(mask.sum())

colors_bin = [ORANGE if e < metric_mean_2wiki else BLUE for e in bin_em]
ax.bar(bin_centers, [e*100 for e in bin_em], width=0.08, color=colors_bin, edgecolor='white', alpha=0.85)
ax.axhline(y=metric_mean_2wiki*100, color='black', linestyle='--', linewidth=1.5,
           label=f'Overall EM={metric_mean_2wiki*100:.1f}%')
ax.set_xlabel('P(retrieve) Bin', fontsize=11)
ax.set_ylabel('Mean EM (%)', fontsize=11)
ax.set_title('EM per Retrieval Confidence Bin\n(2WikiMultihopQA)', fontsize=12, fontweight='bold')
ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda x, _: f'{x:.0f}%'))
ax.legend(fontsize=10)

# Add sample count annotations
for xc, yv, n in zip(bin_centers, bin_em, bin_n):
    if n > 0:
        ax.text(xc, yv*100 + 1.5, f'n={n}', ha='center', fontsize=7, color='#333')

plt.tight_layout()
plt.savefig(f'{FIGDIR}/fig2_proba_r_analysis.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved fig2_proba_r_analysis.png')

In [ ]:
#  Figure 3: Threshold Sensitivity Analysis 
# This figure uses the simulated_em data from Section 4

thresholds_plot = [r['thresh'] for r in simulated_em]
em_ret_plot     = [r['em_if_retrieve'] for r in simulated_em]
n_ret_plot      = [r['would_retrieve']  for r in simulated_em]

fig, ax1 = plt.subplots(figsize=(9, 5))
ax2 = ax1.twinx()

line1, = ax1.plot(thresholds_plot, [e*100 if not np.isnan(e) else 0 for e in em_ret_plot],
                  color=BLUE, linewidth=2.5, marker='o', markersize=6, label='EM (retrieved samples)')
line2, = ax2.plot(thresholds_plot, n_ret_plot, color=GREEN, linewidth=2,
                  linestyle='--', marker='s', markersize=5, label='# Samples that would retrieve')

ax1.axvline(x=0.5, color='red', linestyle=':', linewidth=2, label='Default threshold=0.5')
ax1.set_xlabel('Retrieval Threshold', fontsize=12)
ax1.set_ylabel('Exact Match (%) — retrieved samples', fontsize=11, color=BLUE)
ax2.set_ylabel('# Samples above threshold', fontsize=11, color=GREEN)
ax1.set_title('Threshold Sensitivity Analysis\n(2WikiMultihopQA — post-hoc on existing proba_r)', fontsize=13, fontweight='bold')
ax1.tick_params(axis='y', labelcolor=BLUE)
ax2.tick_params(axis='y', labelcolor=GREEN)
ax1.yaxis.set_major_formatter(plt.FuncFormatter(lambda x, _: f'{x:.0f}%'))

lines = [line1, line2,
         plt.Line2D([0],[0], color='red', linestyle=':', linewidth=2, label='Default threshold=0.5')]
ax1.legend(handles=lines, loc='lower left', fontsize=10)
plt.tight_layout()
plt.savefig(f'{FIGDIR}/fig3_threshold_sensitivity.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved fig3_threshold_sensitivity.png')

In [ ]:
#  Figure 4: Retrieval Confidence Stratification 

proba_r = np.array(proba_r_2wiki)
em_arr  = np.array(em_2wiki)

labels  = ['High-conf\nRetrieve\n(P > 0.7)', 'Ambiguous\n(0.3 ≤ P ≤ 0.7)', 'High-conf\nSkip\n(P < 0.3)']
masks   = [proba_r > 0.7, (proba_r >= 0.3) & (proba_r <= 0.7), proba_r < 0.3]
colors_s = [BLUE, ORANGE, GREEN]

em_per_stratum = [em_arr[m].mean() * 100 if m.sum() > 0 else 0 for m in masks]
n_per_stratum  = [m.sum() for m in masks]

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 5))

# Sample distribution
wedges, texts, autotexts = ax1.pie(
    n_per_stratum, labels=labels, colors=colors_s,
    autopct='%1.1f%%', startangle=140,
    textprops={'fontsize': 10})
ax1.set_title('Sample Distribution by\nRetrieval Confidence', fontsize=12, fontweight='bold')

# EM per stratum
bars = ax2.bar(labels, em_per_stratum, color=colors_s, edgecolor='white', alpha=0.85, width=0.55)
for bar, val, n in zip(bars, em_per_stratum, n_per_stratum):
    ax2.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.8,
             f'{val:.1f}%\n(n={n})', ha='center', va='bottom', fontsize=10, fontweight='bold')
ax2.axhline(y=metric_mean_2wiki*100, color='black', linestyle='--', linewidth=1.5,
            label=f'Overall EM={metric_mean_2wiki*100:.1f}%')
ax2.set_ylabel('Exact Match (%)', fontsize=11)
ax2.set_title('EM per Retrieval Confidence Stratum\n(2WikiMultihopQA)', fontsize=12, fontweight='bold')
ax2.set_ylim(0, 85)
ax2.yaxis.set_major_formatter(plt.FuncFormatter(lambda x, _: f'{x:.0f}%'))
ax2.legend(fontsize=10)
ax2.grid(axis='y', alpha=0.4)

plt.tight_layout()
plt.savefig(f'{FIGDIR}/fig4_confidence_stratification.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved fig4_confidence_stratification.png')

In [ ]:
#  Figure 5: Threshold Sweep Results 

if len(thresh_results) >= 3:
    thresholds_sw = sorted(thresh_results.keys())
    em_sw = [thresh_results[t]['em'] * 100 for t in thresholds_sw]

    fig, ax = plt.subplots(figsize=(7, 5))
    ax.plot(thresholds_sw, em_sw, marker='D', markersize=10,
            linewidth=2.5, color=BLUE)
    for t, e in zip(thresholds_sw, em_sw):
        ax.annotate(f'{e:.1f}%', (t, e), textcoords='offset points',
                    xytext=(0, 12), ha='center', fontsize=11, fontweight='bold')
    ax.set_xlabel('Retrieval Threshold', fontsize=12)
    ax.set_ylabel('Exact Match (%)', fontsize=12)
    ax.set_title('EM vs Retrieval Threshold\n(2WikiMultihopQA — Adaptive Retrieval)', fontsize=13, fontweight='bold')
    ax.set_ylim(0, 85)
    ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda x, _: f'{x:.0f}%'))
    ax.grid(alpha=0.4)
    plt.tight_layout()
    plt.savefig(f'{FIGDIR}/fig5_threshold_sweep.png', dpi=150, bbox_inches='tight')
    plt.show()
    print('Saved fig5_threshold_sweep.png')
else:
    print('Bonus experiments not run — skipping Figure 5.')

---
## Section 6 — Summary Table

In [ ]:
print('=' * 70)
print(' COMPLETE RESULTS SUMMARY')
print('=' * 70)
print(f'{"Experiment":<40} {"Dataset":<22} {"EM":>8}')
print('-' * 70)
rows = [
    ('Paper (OPEN-RAG)',               '2WikiMultihopQA', PAPER_2WIKI_EM),
    ('Reproduced (A2, thresh=0.5)',    '2WikiMultihopQA', metric_mean_2wiki),
    ('Paper (OPEN-RAG)',               'HotpotQA',        PAPER_HOTPOT_EM),
    ('New dataset eval (A3)',          'HotpotQA',        em_hotpot),
]
# Add threshold sweep if available
for t, r in sorted(thresh_results.items()):
    if t != 0.5:
        rows.append((f'Threshold sweep (thresh={t})', '2WikiMultihopQA', r['em']))

for exp, ds, em in rows:
    print(f'{exp:<40} {ds:<22} {em*100:>7.2f}%')
print('=' * 70)

# Save summary to JSON
summary = {
    'paper_2wiki_em':   PAPER_2WIKI_EM,
    'our_2wiki_em':     metric_mean_2wiki,
    'paper_hotpot_em':  PAPER_HOTPOT_EM,
    'our_hotpot_em':    em_hotpot,
    'threshold_sweep':  {str(t): r['em'] for t, r in thresh_results.items()},
    'n_2wiki':          len(em_2wiki),
    'n_hotpot':         n_hotpot,
}
with open('/content/openrag/eval/results_summary.json', 'w') as f:
    json.dump(summary, f, indent=2)
print('Summary saved to /content/openrag/eval/results_summary.json')


## Section 7 — Download All Outputs


In [ ]:
import shutil
shutil.make_archive('/content/assignment3_outputs', 'zip', '/content/openrag/eval')
print('Created /content/assignment3_outputs.zip')
print('\nDownload it via: Files panel → right-click → Download')

# List all output files
import glob
print('\nAll output files:')
for f in sorted(glob.glob('/content/openrag/eval/**/*', recursive=True)):
    if os.path.isfile(f):
        size = os.path.getsize(f)
        print(f'  {f.replace("/content/openrag/eval/","")} ({size/1024:.1f} KB)')